In [2]:
# -*- Encoding:UTF-8 -*-
from sklearn.metrics import accuracy_score, recall_score, f1_score
class Evaluation(object):
    def __init__(self):
        pass

    # get top predicted genes
    @staticmethod
    def get_top_genes(gene_list, score_list, label_list):
        n_gene = len(gene_list)
        score_gene = []
        hit_list = []
        gene_label = {}  # key: gene, value: rating
        # score_list为预测结果
        for i in range(n_gene):
            # 将rating与基因对应
            score_gene.append([score_list[i], gene_list[i]])
            # 将基因与标签对应
            gene_label[gene_list[i]] = label_list[i]
        # 将预测基因结果进行排序，选择score靠前的基因
        score_gene.sort(reverse=True)
        # 选择排名靠前1000的基因
        for score, gene in score_gene[0:int(sum(label_list))]:
            # 如果测试基因中预测排名靠前的基因对应的label==1，则hit_list添加1，否则表示预测错了，添加0
            # 表示预测对了，因为若与该疾病关联概率（rating）较大的基因的标签为1表明疾病与该基因之间确实有关联，则预测正确
            if gene_label[gene] == 1.0:
                hit_list.append(1)
            else:
                hit_list.append(0)
        #计算测试集中该疾病对应相关基因个数
        n_known_genes = int(sum(label_list))
        #排名靠前的队列中预测对的相关基因个数
        n_topk_hit = sum(hit_list[:n_known_genes])
        return hit_list, n_known_genes, n_topk_hit
# n_total_test为测试集中与该疾病有关的基因个数
    def cal_prf(self, all_hit_list, n_total_test):
        # 排名前3,5,10
        top_k_list = [5, 10, 15,20]
        all_hit_num_list = []
        test_dis_num = len(all_hit_list)
#         预测标签
        all_pre_list = []
        prf_summary = []
        for i in range(test_dis_num):
            # 0或1的列表，代表一种疾病的所有测试集基因
            hit_list = all_hit_list[i]
            hit_num_list = []
#             每一个疾病的前3/5/10预测标签
            hit_pre_list_3=[]
            hit_pre_list_5=[]
            hit_pre_list_10=[]
            for k in top_k_list:
                #score排名前k个中猜中的个数
                hit_num_list.append(sum(hit_list[:k]))

#             hit_pre_list_3.append(hit_list[:3])
#             hit_pre_list_5.append(hit_list[:5])
#             hit_pre_list_10.append(hit_list[:10])
            # 取所有疾病对应的排名前3,5,10的列表，记录前3,5,10的列表中相关基因预测正确的有多少
            all_hit_num_list.append(hit_num_list)
#         all_pre_list_3 = [item for sublist in hit_pre_list_3 for item in sublist]
#         all_pre_list_5 = [item for sublist in hit_pre_list_5 for item in sublist]
#         all_pre_list_10 = [item for sublist in hit_pre_list_10 for item in sublist]

#             all_pre_list.append(hit_pre_list)

        for i in range(len(top_k_list)):
            # 3/5/10
            top_k = top_k_list[i]
# 包括每一种疾病的前k个排名基因预测正确的个数
            temp = [x[i] for x in all_hit_num_list]
            # 对于所有疾病：排名前3/5/10的列表中预测对的相关基因的总个数
            hit_sum = sum(temp)
#             所有疾病猜中
            precision = (hit_sum*1.0) / (top_k * test_dis_num)
            recall = (hit_sum*1.0) / n_total_test
            f1 = self.cal_f1(precision, recall)
            prf_summary.append(precision)
            prf_summary.append(recall)
            prf_summary.append(f1)
# 明天将这一块的代码替换成用包计算的准确度那些
#         all_true_list_3 =  [1] * len(all_pre_list_3)
#         all_true_list_5 =  [1] * len(all_pre_list_5)
#         all_true_list_10 =  [1] * len(all_pre_list_10)
#         accuracy_3 = accuracy_score(all_true_list_3, all_pre_list_3)
#         recall_3 = recall_score(all_true_list_3, all_pre_list_3)
#         f1_3 = f1_score(all_true_list_3, all_pre_list_3)
#         prf_summary.append(accuracy_3)
#         prf_summary.append(recall_3)
#         prf_summary.append(f1_3)
#         accuracy_5 = accuracy_score(all_true_list_5, all_pre_list_5)
#         recall_5 = recall_score(all_true_list_5, all_pre_list_5)
#         f1_5 = f1_score(all_true_list_5, all_pre_list_5)
#         prf_summary.append(accuracy_5)
#         prf_summary.append(recall_5)
#         prf_summary.append(f1_5)
#         accuracy_10 = accuracy_score(all_true_list_10, all_pre_list_10)
#         recall_10 = recall_score(all_true_list_10, all_pre_list_10)
#         f1_10 = f1_score(all_true_list_10, all_pre_list_10)
#         prf_summary.append(accuracy_10)
#         prf_summary.append(recall_10)
#         prf_summary.append(f1_10)
        return prf_summary

    def cal_metrics(self, gene_array, score_array, rating):
        n_gene = len(gene_array)
        # print('n_gene_array:', n_gene)
        # print('n_score_array:', len(score_array))
        score_gene = []
        hit_list = []
        gene_rating = {}  # key: gene, value: rating
        for i in range(n_gene):
            score_gene.append([score_array[i], gene_array[i]])
            gene_rating[gene_array[i]] = rating[i]
        score_gene.sort(reverse=True)
        for score, gene in score_gene:
            if gene_rating[gene] == 1.0:
                hit_list.append(1)
            else:
                hit_list.append(0)
        known_gene_num = int(sum(rating))
        prf = self.cal_pr_re_f1(hit_list, known_gene_num)  # prf means precision, recall, f1-measure
        topk_hit = sum(hit_list[:known_gene_num])

        return prf, topk_hit

    @staticmethod
    def cal_prf_avg(prf_array):
        prf_avg = list()
        array_num = len(prf_array)
        for i in range(len(prf_array[0])):
            temp = sum(prf_array[:, i]) / array_num
            prf_avg.append(temp)
        return prf_avg

    # calculate pr, re, f1
    def cal_pr_re_f1(self, hit_list, known_gene_num):
        top_k_list = [5, 10, 15,20]
        prf_list = []
        for k in top_k_list:
            topk_pr = sum(hit_list[:k]) / float(k)
            topk_re = sum(hit_list[:k]) / float(known_gene_num)
            topk_f1 = self.cal_f1(topk_pr, topk_re)
            prf_list.append(topk_pr)
            prf_list.append(topk_re)
            prf_list.append(topk_f1)
        return prf_list

    # calculate F-measure
    @staticmethod
    def cal_f1(precision, recall):
        if precision == 0 or recall == 0: return 0
        return precision * recall * 2 / (precision + recall)


In [3]:
#gat

In [4]:
import tensorflow as tf

class GraphAttentionLayer(tf.keras.layers.Layer):
    def __init__(self, in_features, out_features, dropout, alpha, concat=True):
        super(GraphAttentionLayer, self).__init__()
        self.dropout = dropout
        self.in_features = in_features
        self.out_features = out_features
        self.alpha = alpha
        self.concat = concat

        with tf.variable_scope(self.name):
            self.W = tf.get_variable(
                name='weight',
                shape=(in_features, out_features),
                initializer=tf.truncated_normal_initializer(mean=0.0, stddev=0.01),
                dtype=tf.float32)
            self.a = tf.get_variable(
                name='bias',
                shape=(2*out_features,1),
                initializer=tf.truncated_normal_initializer(mean=0.0, stddev=0.01),
                dtype=tf.float32)
        self.vars = [self.W]


    def call(self, h, adj):
        Wh = tf.matmul(h, self.W)
        e = self._prepare_attentional_mechanism_input(Wh)

        zero_vec = -9e15 * tf.ones_like(e)
        attention = tf.where(adj > 0, e, zero_vec)
        softmax_denominator = np.sum(attention, axis=0, keepdims=True)
        attention = attention / softmax_denominator

        #attention = tf.nn.softmax(attention, axis=1)
        #axis = 1 if attention.shape.ndims == 2 else -1
        #attention = tf.nn.softmax(attention, axis=axis)
        attention = tf.nn.dropout(attention, rate=self.dropout)
        h_prime = tf.matmul(tf.transpose(attention), Wh)

        if self.concat:
            return tf.nn.elu(h_prime)
        else:
            return h_prime

    def _prepare_attentional_mechanism_input(self, Wh):
        #128,1      
        Wh1 = tf.matmul(Wh, self.a[:self.out_features, :])
        # 128,1
        Wh2 = tf.matmul(Wh, self.a[self.out_features:, :])
        # 128,128
        e = Wh1 + tf.transpose(Wh2)
        return tf.nn.leaky_relu(e, alpha=self.alpha)

/opt/conda/lib/python3.7/site-packages/tensorflow/python/framework/dtypes.py:526: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
/opt/conda/lib/python3.7/site-packages/tensorflow/python/framework/dtypes.py:527: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
/opt/conda/lib/python3.7/site-packages/tensorflow/python/framework/dtypes.py:528: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
/opt/conda/lib/python3.7/site-packages/tensorflow/python/framework/dtypes.py:529: FutureWarning: Passing (type, 1) or 

In [5]:
#gcn

In [6]:
import tensorflow as tf
# from tensorflow.keras import layers, initializers
class GraphConvolution(tf.keras.layers.Layer):
    def __init__(self, in_features, out_features):
        super(GraphConvolution, self).__init__()
        # self.weight = self.add_weight(
        #     name="weight",
        #     shape=(in_features, out_features),
        #     initializer=initializers.GlorotUniform(),
        #     trainable=True,
        # )
        # self.bias = self.add_weight(
        #     name="bias", shape=(out_features,), initializer="zeros", trainable=True
        # )
        with tf.variable_scope(self.name):
            self.weight = tf.get_variable(
                name='weight',
                shape=(in_features, out_features),
                initializer=tf.truncated_normal_initializer(mean=0.0, stddev=0.01),
                dtype=tf.float32)
            self.bias = tf.get_variable(
                name='bias',
                shape=out_features,
                initializer=tf.truncated_normal_initializer(mean=0.0, stddev=0.01),
                dtype=tf.float32)
        self.vars = [self.weight]

    def call(self, x, adj):
        #       batchsize,out_features
        support = tf.matmul(x, self.weight)
        #         batchsize,out_features
        output = tf.matmul(adj, support) + self.bias
        return output
    
class GraphConvolution1(tf.keras.layers.Layer):
    def __init__(self, in_features, out_features):
        super(GraphConvolution1, self).__init__()
        # self.weight = self.add_weight(
        #     name="weight",
        #     shape=(in_features, out_features),
        #     initializer=initializers.GlorotUniform(),
        #     trainable=True,
        # )
        # self.bias = self.add_weight(
        #     name="bias", shape=(out_features,), initializer="zeros", trainable=True
        # )
        with tf.variable_scope(self.name):
            self.weight = tf.get_variable(
                name='weight',
                shape=(in_features, out_features),
                initializer=tf.truncated_normal_initializer(mean=0.0, stddev=0.01),
                dtype=tf.float32)
            self.bias = tf.get_variable(
                name='bias',
                shape=out_features,
                initializer=tf.truncated_normal_initializer(mean=0.0, stddev=0.01),
                dtype=tf.float32)
        self.vars = [self.weight]

    def call(self, x, adj):
        #       batchsize,out_features
        support = tf.matmul(x, self.weight)
        #         batchsize,out_features
        output = tf.matmul(adj, support) + self.bias
        #     batchsize,hidden_features1
        # 首先经过普通的GraphConvolution得到hidden_feature维度输出，然后经过第二个卷积时在最后一步借助Dense思想翻转，
        # 使得最后输出维度还是hidden_feature1
        output = tf.matmul(output, tf.transpose(self.weight))
        return output

# class GCN(tf.keras.Model):
#     def __init__(self, in_features, hidden_features1,hidden_features2, out_features):
#         super(GCN, self).__init__()
#         self.gc1 = GraphConvolution(in_features, hidden_features1)
#         self.gc2 = GraphConvolution(hidden_features1, out_features)

#     def call(self, x, adj):
#         x = tf.nn.relu(self.gc1(x, adj))
#         x = self.gc2(x, adj)
#         return x
# 删去一层
class GCN(tf.keras.Model):
    def __init__(self, in_features, hidden_features1, out_features):
        super(GCN, self).__init__()
        self.gc1 = GraphConvolution(in_features, hidden_features1)
        self.gc2 = GraphConvolution(hidden_features1, out_features)

    def call(self, x, adj):
        x = tf.nn.relu(self.gc1(x, adj))
        x = self.gc2(x, adj)
        return x
# class GCN1(tf.keras.Model):
#     def __init__(self, in_features, hidden_features1, out_features):
#         super(GCN1, self).__init__()
#         self.gc1 = GraphConvolution(in_features, hidden_features1)
#         self.gc2 = GraphConvolution1(hidden_features1, out_features)

#     def call(self, x, adj):
#         x = tf.nn.relu(self.gc1(x, adj))
#         #batch,out_features
#         #x = tf.nn.dropout(x, 1)
#         x = self.gc2(x, adj)
#         return x

In [7]:
# 添加了GAT
class GCN1(tf.keras.Model):
    def __init__(self, in_features, hidden_features1,nemb, out_features,dropout,alpha,nheads):
    #def __init__(self, in_features, hidden_features1,out_features):
        super(GCN1, self).__init__()
        self.gc1 = GraphConvolution(in_features, hidden_features1)
        #self.gat = GraphAttentionLayer(hidden_features1, hidden_features2, dropout=dropout, alpha=alpha, concat=True)
        #batch,hidden_features1
        #self.gc2 = GraphConvolution1(hidden_features1, out_features)
        self.gc2 = GraphConvolution(hidden_features1, out_features)
        #self.attentions = [GraphAttentionLayer(in_features, hidden_features2, dropout=dropout, alpha=alpha, concat=True) for _ in range(nheads)]  
        
        #self.attentions = [GraphAttentionLayer(nfeat, nhid, dropout=dropout, alpha=alpha, concat=True)]
        #self.out_att = GraphAttentionLayer(hidden_features2 * nheads, nemb, dropout=dropout, alpha=alpha, concat=False)
        self.out_att = GraphAttentionLayer(out_features, nemb, dropout=dropout, alpha=alpha, concat=False)

    def call(self, x, adj):
        x = tf.nn.relu(self.gc1(x, adj))
        #batch,out_features
        x = self.gc2(x, adj)
        #x = tf.nn.dropout(x, rate=0.2)
        #x = tf.concat([att(x, adj) for att in self.attentions], axis=1)
        #x = tf.nn.dropout(x, rate=0)
        #x = tf.nn.elu(self.out_att(x, adj))
        return x

In [8]:
# @tf.function
def cosine_similarity(X):
    # 计算余弦相似度矩阵
    norm = tf.linalg.norm(X, axis=1, keepdims=True)
    similarity_matrix = tf.matmul(X, X, transpose_b=True) / (norm * tf.transpose(norm))

    return similarity_matrix

# Construct adjacency matrix and add identity matrix

def construct_adjacency_matrix(similarity_matrix, threshold):
    # Create identity matrix
    identity_matrix = tf.cast(tf.eye(num_rows, num_columns, dtype=tf.float32), dtype=tf.int32)
    # Add identity matrix to the adjacency matrix
    adjacency_matrix_np = adjacency_matrix
#     adjacency_matrix_np = adjacency_matrix + identity_matrix

    return adjacency_matrix_np

In [9]:
import tensorflow as tf
from abc import abstractmethod
LAYER_IDS = {}

def get_layer_id(layer_name=''):
    if layer_name not in LAYER_IDS:
        LAYER_IDS[layer_name] = 0
        return 0
    else:
        LAYER_IDS[layer_name] += 1
        return LAYER_IDS[layer_name]


class Layer(object):
    def __init__(self, name):
        if not name:
            layer = self.__class__.__name__.lower()
            name = layer + '_' + str(get_layer_id(layer))
        self.name = name
        self.vars = []

    def __call__(self, inputs):
        outputs = self._call(inputs)
        return outputs

    @abstractmethod
    def _call(self, inputs):
        pass


class Dense(Layer):
    def __init__(self, input_dim, output_dim, dropout=0.8, act=tf.nn.relu, name=None):
        super(Dense, self).__init__(name)
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.dropout = dropout
        # relu函数
        self.act = act

        with tf.variable_scope(self.name):
            self.weight = tf.get_variable(
                name='weight',
                shape=(input_dim, output_dim),
                initializer=tf.truncated_normal_initializer(mean=0.0, stddev=0.01),
                dtype=tf.float32)
            self.bias = tf.get_variable(
                name='bias',
                shape=output_dim,
                initializer=tf.truncated_normal_initializer(mean=0.0, stddev=0.01),
                dtype=tf.float32)
        self.vars = [self.weight]
    # 利用网络层类对象进行前向计算时，只需要调用类的 __call__ 方法即可，即写成 model(x) 方式便可
    def _call(self, inputs):
        x = tf.nn.dropout(inputs, 1-self.dropout)
        output = tf.matmul(x, self.weight) + self.bias
        return self.act(output)


In [10]:
# -*- Encoding:UTF-8 -*-
import tensorflow as tf
from tensorflow.keras.losses import BinaryCrossentropy
tf.reset_default_graph()
# from layers_gcn import GCN
#Dense层变成GCN层
class Model(object):
    def __init__(self, args, data):

        self._add_placeholders()
        self._add_model(args, data)
        self._add_loss(args)
        self._add_train(args)

    def _add_placeholders(self):
        self.dis = tf.placeholder(tf.int32)
        self.gene = tf.placeholder(tf.int32)
        self.dg_label = tf.placeholder(tf.float32)

        # for computing l2 loss
        self.vars_dis = []
        self.vars_gene = []

    def _add_model(self, args, data):

        dis_matrix1 = data[0]
        gene_matrix1 = data[1]
        n_dis_dim = len(dis_matrix1[0])
        n_gene_dim = len(gene_matrix1[0])

        dis_symp_matrix = data[4]
        gene_go_matrix = data[5]
        n_symp = len(dis_symp_matrix[0])
        n_go = len(gene_go_matrix[0])

        self.dis_matrix = tf.convert_to_tensor(dis_matrix1)
        self.gene_matrix = tf.convert_to_tensor(gene_matrix1)
        self.dis_matrix_s = tf.convert_to_tensor(dis_symp_matrix)
        self.gene_matrix_s = tf.convert_to_tensor(gene_go_matrix)

        self.dis_embedding = tf.nn.embedding_lookup(self.dis_matrix, self.dis)
        self.gene_embedding = tf.nn.embedding_lookup(self.gene_matrix, self.gene)
        self.dis_embedding_s = tf.nn.embedding_lookup(self.dis_matrix_s, self.dis)
        self.gene_embedding_s = tf.nn.embedding_lookup(self.gene_matrix_s, self.gene)
        
        similarity_matrix = cosine_similarity(self.dis_embedding)
        # 设置阈值
        threshold = 0.6
        # 构建邻接矩阵
        self.dis_adj = construct_adjacency_matrix(similarity_matrix, threshold)
        
        self.dis_adj=tf.cast(self.dis_adj,dtype=tf.float32)
        
        similarity_matrix = cosine_similarity(self.gene_embedding)
        # 设置阈值
        threshold = 0.6
        # 构建邻接矩阵
        self.gene_adj = construct_adjacency_matrix(similarity_matrix, threshold)
        self.gene_adj=tf.cast(self.gene_adj,dtype=tf.float32)
        
        similarity_msatrix = cosine_similarity(self.dis_embedding_s)
        # 设置阈值
        threshold = 0.6
        # 构建邻接矩阵
        self.dis_s_adj = construct_adjacency_matrix(similarity_matrix, threshold)
        self.dis_s_adj=tf.cast(self.dis_s_adj,dtype=tf.float32)
        
        similarity_matrix = cosine_similarity(self.gene_embedding_s)
        # 设置阈值
        threshold= 0.6
        # 构建邻接矩阵
        self.gene_s_adj = construct_adjacency_matrix(similarity_matrix, threshold)
        self.gene_s_adj=tf.cast(self.gene_s_adj,dtype=tf.float32)
        # self.num_nodes=0
        # self.node_features = tf.random.normal((10, args.disLayer[0]))

        # disease vector based on genes
        with tf.name_scope("Dis_layer"):
            dis_w1 = self.init_variable([n_dis_dim, args.disLayer[0]],"dis_w1")
            self.dis_embedding = tf.matmul(self.dis_embedding, dis_w1)
            for i in range(len(args.disLayer) - 1):
                dis_mlp = GCN(in_features=args.disLayer[0], hidden_features1=args.dim[0],out_features=args.disLayer[i+1])
#                 dis_mlp = GCN1(in_features=args.disLayer[0], 
#                                  hidden_features1=args.dim[0],
#                                  #hidden_features2=args.dim[1],
#                                  nemb=args.dim[1],
#                                  out_features=args.disLayer[i+1],
#                                  dropout=0.2, 
#                                  alpha=0.3,
#                                  nheads=2
#                                 )
                # num_nodes = tf.shape(self.dis_embedding)[0]
                # print(num_nodes)
                # node_features_dis = tf.random.normal((num_nodes, args.disLayer[i]))
                self.dis_embedding = dis_mlp(self.dis_embedding,self.dis_adj)
#                 self.vars_dis.extend(dis_mlp.gc1.vars)
#                 self.vars_dis.extend(dis_mlp.gc2.vars)
#                 self.vars_dis.extend(dis_mlp.out_att.vars)
        #
        # # disease vector based on symptoms
        with tf.name_scope("Dis_symp_layer"):
            dis_w1_s = self.init_variable([n_symp, args.disLayer_s[0]],"dis_w1_s")
            self.dis_embedding_s = tf.matmul(self.dis_embedding_s, dis_w1_s)
            '''
            for i in range(len(args.disLayer_s) - 1):
            dis_mlp_s = Dense(input_dim=args.disLayer_s[i], output_dim=args.dims)
            self.dis_embedding_s = dis_mlp_s(self.dis_embedding_s)
            self.vars_dis.extend(dis_mlp_s.vars)
            '''
            for i in range(len(args.disLayer_s) - 1):
                dis_mlp_s = GCN(in_features=args.disLayer_s[0], hidden_features1=args.dims[0],out_features=args.disLayer_s[i+1])
#                 dis_mlp_s = GCN1(in_features=args.disLayer_s[0], 
#                                hidden_features1=args.dims[0],
#                                #hidden_features2=args.dims[1],
#                                nemb=args.dims[1],
#                                out_features=args.disLayer_s[i+1],
#                                dropout=0.2, 
#                                alpha=0.3,
#                                nheads=2
#                               )
                self.dis_embedding_s = dis_mlp_s(self.dis_embedding_s,self.dis_s_adj)
#                 self.vars_dis.extend(dis_mlp_s.gc1.vars)
#                 self.vars_dis.extend(dis_mlp_s.gc2.vars)
#                 self.vars_dis.extend(dis_mlp_s.out_att.vars)
        #
        # # gene vector based on diseases
        with tf.name_scope("Gene_layer"):
            gene_w1 = self.init_variable([n_gene_dim, args.geneLayer[0]],"gene_w1")
            self.gene_embedding = tf.matmul(self.gene_embedding, gene_w1)
            for i in range(len(args.geneLayer) - 1):
                gene_mlp = GCN(in_features=args.geneLayer[0], hidden_features1=args.dim[0],out_features=args.geneLayer[i+1])
#                 gene_mlp = GCN1(in_features=args.geneLayer[0], 
#                                  hidden_features1=args.dim[0],
#                                  #hidden_features2=args.dim[1],
#                                  nemb=args.dim[1],
#                                  out_features=args.geneLayer[i+1],
#                                  dropout=0.2, 
#                                  alpha=0.3,
#                                  nheads=2
#                                 )
                # num_nodes = tf.shape(self.gene_embedding)[0]
                # print(num_nodes)
                # node_features_gene = tf.random.normal((num_nodes, args.disLayer[i]))
                self.gene_embedding = gene_mlp(self.gene_embedding,self.gene_adj)
#                 self.vars_gene.extend(gene_mlp.gc1.vars)
#                 self.vars_gene.extend(gene_mlp.gc2.vars)
#                 self.vars_gene.extend(gene_mlp.out_att.vars)
        #
        # # gene vector based on go
        with tf.name_scope("Gene_go_layer"):
            gene_w1_s = self.init_variable([n_go, args.geneLayer_s[0]],"gene_w1_s")
            self.gene_embedding_s = tf.matmul(self.gene_embedding_s, gene_w1_s)
            '''
            for i in range(len(args.geneLayer_s) - 1):
            gene_mlp_s = Dense(input_dim=args.geneLayer_s[i], output_dim=args.dims)
            self.gene_embedding_s = gene_mlp_s(self.gene_embedding_s)
            self.vars_gene.extend(gene_mlp_s.vars)
            '''
            for i in range(len(args.geneLayer_s) - 1):
                gene_mlp_s = GCN(in_features=args.geneLayer_s[0], hidden_features1=args.dims[0],out_features=args.geneLayer_s[i+1])
#                 gene_mlp_s = GCN1(in_features=args.geneLayer_s[0], 
#                                 hidden_features1=args.dims[0],
#                                 #hidden_features2=args.dims[1],
#                                 nemb=args.dims[1],
#                                 out_features=args.geneLayer_s[i+1],
#                                 dropout=0.2, 
#                                 alpha=0.3,
#                                 nheads=264
#                                )
                
                self.gene_embedding_s = gene_mlp_s(self.gene_embedding_s,self.gene_s_adj)
#                 self.vars_gene.extend(gene_mlp_s.gc1.vars)
#                 self.vars_gene.extend(gene_mlp_s.gc2.vars)
#                 self.vars_gene.extend(gene_mlp_s.out_att.vars)
                

        #concat dis_embedding and dis_embedding_s
#         self.dis_embedding = tf.concat([self.dis_embedding, self.dis_embedding_s], axis=1)
        #
        # concat gene_embedding and gene_embedding_s
        self.gene_embedding = tf.concat([self.gene_embedding, self.gene_embedding_s], axis=1)
        #
        norm_dis_output = tf.sqrt(tf.reduce_sum(tf.square(self.dis_embedding), axis=1))
        norm_gene_output = tf.sqrt(tf.reduce_sum(tf.square(self.gene_embedding), axis=1))
        #
        # # y of dis-gene
        multiply_dg = tf.reduce_sum(tf.multiply(self.dis_embedding, self.gene_embedding), axis=1, keep_dims=False)
        self.dg_y = multiply_dg / (norm_dis_output * norm_gene_output)
        self.dg_y = tf.maximum(1e-6, self.dg_y)

    @staticmethod
    def init_variable(shape, name):
        input_dim, output_dim = shape
        return tf.get_variable(
            name=name,
            shape=(input_dim, output_dim),
            initializer=tf.truncated_normal_initializer(mean=0.0, stddev=0.01),
            dtype=tf.float32)

    def _add_loss(self, args):
        # dis-gene loss
        #self.dg_loss = BinaryCrossentropy()(self.dg_label, self.dg_y)
        temp = self.dg_label * tf.log(self.dg_y) + (1 - self.dg_label) * tf.log(1 - self.dg_y + 1e-6)
        self.dg_loss = -tf.reduce_sum(temp)
        self.dg_l2_loss = tf.nn.l2_loss(self.dis_embedding) + tf.nn.l2_loss(self.gene_embedding)
        for var in self.vars_dis:
            self.dg_l2_loss += tf.nn.l2_loss(var)
        for var in self.vars_gene:
            self.dg_l2_loss += tf.nn.l2_loss(var)
        self.dg_loss = self.dg_loss + self.dg_l2_loss * args.l2_weight
#         self.dg_loss = self.dg_loss


    def _add_train(self, args):
        self.optimizer_dg = tf.train.AdamOptimizer(args.lr).minimize(self.dg_loss)

    # train dis-gene network
    def train_dg_net(self, sess, feed_dict):
        return sess.run([self.optimizer_dg, self.dg_loss, self.dg_y], feed_dict)

    # predict dis-gene associations
    def predict_dg(self, sess, feed_dict):
        return sess.run([self.dg_y], feed_dict)





In [13]:
# 在需要时加载保存的变量
import pickle
with open('./data.pkl', 'rb') as file:
    loaded_data = pickle.load(file)

# 提取加载的变量
dis_matrix = loaded_data['dis_matrix']
gene_matrix = loaded_data['gene_matrix']
n_dis = loaded_data['n_dis']
n_gene = loaded_data['n_gene']
train_d = loaded_data['train_d']
train_g = loaded_data['train_g']
l = loaded_data['train_l']
test_d = loaded_data['test_d']
test_g = loaded_data['test_g']
test_l = loaded_data['test_l']
dis_symp_matrix = loaded_data['dis_symp_matrix']
gene_go_matrix = loaded_data['gene_go_matrix']
# loaded_test_data = loaded_data['test_data']

In [14]:
print(train_d)
print(len(train_d))

[333 333 333 ... 313 313 313]
243317


In [15]:
import tensorflow as tf
# import tensorflow.compat.v1 as tf
# tf.compat.v1.disable_eager_execution()
import numpy as np
import argparse
# from DataSet import DataSet
# from Evaluation import Evaluation
import sys
import os
# from model_test4 import Model
# import tensorflow.compat.v1 as tf
# tf.disable_v2_behavior()
class Train:
    def __init__(self):
        os.environ["CUDA_VISIBLE_DEVICES"] = "1"
        self._set_args()
#         self.args.dim = 2048
#         self.args.dims = 1024
#         self.args.disLayer = [4096, 1024]
#         self.args.geneLayer = [4096, 1024]
#         self.args.disLayer_s = [2048, 512]
#         self.args.geneLayer_s = [2048, 512]
        self.args.dim = [512,64]
        self.args.dims = [512,32]
        self.args.disLayer = [256, 192]
        self.args.geneLayer = [1024, 64]
        self.args.disLayer_s = [1024, 64]
        self.args.geneLayer_s = [2048, 128]
#         self.args.disLayer = [1024, 1024]
#         self.args.geneLayer = [1024, 512]
#         self.args.disLayer_s = [1024, 512]
#         self.args.geneLayer_s = [1024, 512]
#         self.args.disLayer = [4096, 512]
#         self.args.geneLayer = [4096, 512]
#         self.args.disLayer_s = [2048, 256]
#         self.args.geneLayer_s = [2048, 256]
        self.args.maxEpochs = 50
        self.args.negNum = 50
        self.args.l2_weight = 1e-5
#         self.data_set = DataSet()
#         self.train()


    def _set_args(self):
        parser = argparse.ArgumentParser(description="Options")
        parser.add_argument('-negNum', action='store', dest='negNum', default=70, type=int)
        parser.add_argument('-dims', action='store', dest='dims', default=64)
        parser.add_argument('-dim', action='store', dest='dim', default=64)
        parser.add_argument('-disLayer', action='store', dest='disLayer', default=[2048, 1024])
        parser.add_argument('-geneLayer', action='store', dest='gendeLayer', default=[2048, 1024])
        parser.add_argument('-disLayer_s', action='store', dest='disLayer_s', default=[1024, 256])
        parser.add_argument('-geneLayer_s', action='store', dest='geneLayer_s', default=[1024, 256])
        parser.add_argument('-lr', action='store', dest='lr', default=0.00001)
        parser.add_argument('-maxEpochs', action='store', dest='maxEpochs', default=20, type=int)
        parser.add_argument('-batchSize', action='store', dest='batchSize', default=50, type=int)
        parser.add_argument('-earlyStop', action='store', dest='earlyStop', default=10)
        parser.add_argument('-checkPoint', action='store', dest='checkPoint', default='./checkPoint/')
        parser.add_argument('-testDisNum', action='store', dest='nTestDis', default=200)
        parser.add_argument('-l2_weight', action='store', dest='l2_weight', default=1e-5)
        parser.add_argument('-simPlus', action='store', dest='simPlus', default=0.2)
        parser.add_argument('-train_interval', action='store', dest='train_interval', default=2)
        self.args = parser.parse_args(args=[])
#         self.args = parser.parse_args()

        # setting gpu
        # self.config = tf.ConfigProto()
        self.config = tf.compat.v1.ConfigProto()
        # self.config.gpu_options.allow_growth = True
        # self.config.allow_soft_placement = True


    def train(self):
        print('train model...')
#         dis_matrix = data_set.dis_matrix
#         gene_matrix = data_set.gene_matrix
#         n_dis = data_set.n_dis
#         n_gene = data_set.n_gene

        data = [dis_matrix, gene_matrix, n_dis, n_gene,
                dis_symp_matrix,
                gene_go_matrix]
        model = Model(self.args, data)
        best_ap = 0
        best_epoch = -1
        best_pr = 0
        with tf.Session(config=self.config) as sess:
            # 目的是初始化所有的全局变量。全局变量包括所有在模型中定义的可训练和不可训练的变量，比如权重矩阵、偏差项等
            sess.run(tf.global_variables_initializer())
            for epoch in range(self.args.maxEpochs):
                # for epoch in range(1):
                print("=" * 20 + "Epoch ", epoch, "=" * 20)
                self.run_epoch(sess, model)
                print('=' * 50)
                print("Start Evaluation!")
                if epoch%1==0:
                    prf_summary, ap, n_total_hit, n_total_test = self.evaluate(sess, model)
                    top5_pr = prf_summary[0]
                    top5_rc = prf_summary[1]
                    top5_f1 = prf_summary[2]
                    top10_pr = prf_summary[3]
                    top10_rc = prf_summary[4]
                    top10_f1 = prf_summary[5]
                    top15_pr = prf_summary[6]
                    top15_rc = prf_summary[7]
                    top15_f1 = prf_summary[8]
                    top20_pr = prf_summary[9]
                    top20_rc = prf_summary[10]
                    top20_f1 = prf_summary[11]
                    print('epoch:', epoch, '; AP:', ap, '; top@5 pr:', top5_pr, '; top@5 rc:', top5_rc,
                     '; top@5 f1:', top5_f1,'; top@10 pr:', top10_pr,'; top@10 rc:', top10_rc,'; top@10 f1:', top10_f1, '; top@15 pr:',
                          top15_pr, '; top@15 rc:', top15_rc,
                     '; top@15 f1:', top15_f1, '; top@20 pr:', top20_pr, '; top@20 rc:', top20_rc,'; top@20 f1:', top20_f1)
                    result = [n_total_test, n_total_hit, ap] + prf_summary
                    result = [str(x) for x in result]
#                 if best_ap < ap and best_pr < top3_pr:
#                     best_ap = ap
#                     best_pr = top3_pr
#                     best_epoch = epoch

#                     print('\t'.join(result))
#                 if epoch - best_epoch > self.args.earlyStop:    # early stop
#                     print("Normal Early stop!")
#                     break
#                 print("=" * 20 + "Epoch ", epoch, "End" + "=" * 20)
            print("Training complete!")


    def run_epoch(self, sess, model):
        # train dis-gene network
        self.train_dg_net(sess, model)

    def train_dg_net(self, sess, model, verbose=1000):

        # train_dis, train_gene, label = self.data_set.train_data
        train_dis, train_gene, label = train_d, train_g, l
        train_len = len(train_dis)
        shuffled_idx = np.random.permutation(np.arange(train_len))
        train_dis = train_dis[shuffled_idx]
        train_gene = train_gene[shuffled_idx]
        label = label[shuffled_idx]

        num_batches = train_len // self.args.batchSize + 1

        losses = []
        #         for i in range(1):
        for i in range(num_batches):
            # if i > 1000: break
            min_idx = i * self.args.batchSize
            max_idx = np.min([train_len, (i + 1) * self.args.batchSize])
            train_d_batch = train_dis[min_idx: max_idx]
            train_g_batch = train_gene[min_idx: max_idx]
            train_l_batch = label[min_idx: max_idx]
            # print(train_d_batch)

            feed_dict = {model.dis: train_d_batch,
                         model.gene: train_g_batch,
                         model.dg_label: train_l_batch}

            #             _, loss, y = model.train_dg_net(sess, feed_dict)
            _, loss, y = model.train_dg_net(sess, feed_dict)
            #             print(y)
#             print(y)
            # print(dis_embedding, dis_embedding.shape)
            # print(gene_embedding, gene_embedding.shape)
            losses.append(loss)
            if verbose and i % verbose == 0:
                sys.stdout.write('\r{} / {} : loss = {}'.format(
                    i, num_batches, np.mean(losses[-verbose:])
                ))
                sys.stdout.flush()
        loss = np.mean(losses)
        print("\nMean loss in DG net is: {}".format(loss))
        return loss

    def evaluate(self, sess, model):
        evaluation = Evaluation()
        test_dis, test_gene, test_label = test_d, test_g, test_l

        n_total_hit = 0.0
        n_total_test = 0.0
        dis_num = len(test_dis)
        all_hit_list = list()
        for i in range(dis_num):
            if i % 100 == 0:
                print(i, '/', dis_num)
            feed_dict = {model.dis: test_dis[i],
                         model.gene: test_gene[i]}
            predict = model.predict_dg(sess, feed_dict)
            # print(len(predict))
            hit_list, n_known_genes, n_topk_hit = evaluation.get_top_genes(test_gene[i], predict[0], test_label[i])
            n_total_hit += n_topk_hit
            n_total_test += n_known_genes
            all_hit_list.append(hit_list)

        ap = n_total_hit / n_total_test
        prf_summary = evaluation.cal_prf(all_hit_list, n_total_test)

        return prf_summary, ap, n_total_hit, n_total_test

In [ ]:
t = Train()
t.train()

train model...
Instructions for updating:
Colocations handled automatically by placer.
Instructions for updating:
keep_dims is deprecated, use keepdims instead
Instructions for updating:
Use tf.cast instead.
====================Epoch  0 ====================
4000 / 4867 : loss = 11.262499809265137
Mean loss in DG net is: 11.337077140808105
Start Evaluation!
0 / 251
100 / 251
200 / 251
epoch: 0 ; AP: 0.7078020315539226 ; top@5 pr: 0.3561752988047809 ; top@5 rc: 0.0966068727036957 ; top@5 f1: 0.15198911934716083 ; top@10 pr: 0.2788844621513944 ; top@10 rc: 0.15128593040847202 ; top@10 f1: 0.1961608518985568 ; top@15 pr: 0.2403718459495352 ; top@15 rc: 0.19559109574238168 ; top@15 f1: 0.2156816015252622 ; top@20 pr: 0.21573705179282868 ; top@20 rc: 0.23406094661767884 ; top@20 f1: 0.2245257593034104
====================Epoch  1 ====================
4000 / 4867 : loss = 9.178148269653322
Mean loss in DG net is: 9.121414184570312
Start Evaluation!
0 / 251
100 / 251
200 / 251
epoch: 1 ; AP: 0